# Enhanced S3 to COG Converter with Chunked Processing

This notebook converts TIF files from S3 to Cloud Optimized GeoTIFFs (COGs) with:
- **Chunked processing** for memory-efficient handling of large files
- **Automatic AWS credential detection** (no .env file needed)
- **Download caching** to avoid re-downloading large files
- **COG validation** before uploading
- **Memory monitoring** and progress tracking

Author: Kyle Lesinger (Enhanced chunked version)

In [1]:
import os
import pandas as pd
import json
import tempfile
import boto3
import rasterio
from rasterio.windows import Window
from rasterio.enums import Resampling
from rasterio.warp import calculate_default_transform, reproject
from rasterio.io import MemoryFile
import rioxarray as rxr
import s3fs
import fsspec
from botocore.exceptions import NoCredentialsError, ClientError
from pathlib import Path
from datetime import datetime
import time
import numpy as np
import gc
import psutil
from tqdm import tqdm

print("✅ Libraries imported successfully!")
print(f"Boto3 version: {boto3.__version__}")
print(f"Rasterio version: {rasterio.__version__}")

✅ Libraries imported successfully!
Boto3 version: 1.37.3
Rasterio version: 1.4.3


In [2]:
# Add path for importing custom modules
import sys
from pathlib import Path

# Add the scripts directory to the Python path
scripts_dir = Path('../scripts').resolve()
if str(scripts_dir) not in sys.path:
    sys.path.insert(0, str(scripts_dir))

# Import functions from list_s3crawler_files module
from list_s3crawler_files import (
    load_drcs_data,
    get_tif_files_from_path,
    get_files_with_full_paths,
    list_available_directories
)

# Import COG and cache utilities
from cog_utilities import (
    check_cache_status,
    clear_cache,
    validate_cog,
    export_COG_PROFILE
)

# Import AWS S3 utilities
from aws_s3_utils import (
    initialize_s3_client,
    verify_s3_client,
    get_all_s3_keys
)

# Import batch processing utilities
from batch_processing import (
    process_file_batch,
    print_batch_summary
)

from memory_utils import (
    get_memory_usage,
    calculate_optimal_chunk_size,
    estimate_chunk_memory,
    format_bytes

)

from convert_utilities import (
    convert_to_proper_CRS_and_cogify_chunked
)
    
print("✅ Custom modules imported successfully!")
print(f"   Module path: {scripts_dir}")

✅ Memory monitoring utilities loaded
✅ Custom modules imported successfully!
   Module path: /home/jovyan/conversion_scripts/convert-files-and-move/scripts


# Useful links
<a href="https://data.disasters.openveda.cloud/browseui/browseui/#drcs_activations/" target="_blank" rel="noopener noreferrer" style="color: blue; font-size: 20px;">drcs_activations OLD Directory</a> -- You can view old directory file structure here.

<a href="https://docs.openveda.cloud/user-guide/content-curation/dataset-ingestion/file-preparation.html" target="_blank" rel="noopener noreferrer" style="color: blue; font-size: 20px;">VEDA docs for file naming conventions</a> -- Helps for understanding why/how we name content.

## List of new 2nd level directories

    "Sentinel-1"
    "Sentinel-2"
    "Landsat"
    "MODIS"
    "VIIRS"
    "ASTER"
    "MASTER"
    "ECOSTRESS"
    "Planet"
    "Maxar"
    "HLS"
    "IMERG"
    "GOES"
    "SMAP"
    "ICESat"
    "GEDI"
    "COMSAR"
    "UAVSAR"
    "WB-57"

In [3]:
# DO NOT CHANGE
DIR_OLD_BASE = 'drcs_activations'
DIR_NEW_BASE = 'drcs_activations_new'
BUCKET = 'nasa-disasters'

In [4]:

EVENT_NAME = '202401_SevereWx_SoutheastUS'  #find the name within drcs_activations OLD Directory (see link above)
PRODUCT_NAME = 'landsat'      #find the name within drcs_activations OLD Directory (see link above)
PATH_OLD = f'{DIR_OLD_BASE}/{EVENT_NAME}/{PRODUCT_NAME}'  # Updated to use actual available directory

In [5]:
# Define COG profile for rasterio (DO NOT CHANGE)
COG_PROFILE = export_COG_PROFILE()

# Chunked processing configuration
CHUNK_CONFIG = {
    "default_chunk_size": 1024,  # Default chunk size in pixels
    "memory_limit_mb": 500,      # Memory limit per chunk in MB
    "show_progress": True,       # Show progress bars
    "enable_memory_monitoring": True  # Monitor memory usage
}

## Initialize AWS S3 Client with automatic credential detection

In [6]:
# Initialize AWS S3 Client using the imported function
s3_client, fs_read = initialize_s3_client(bucket_name=BUCKET, verbose=True)

# Verify S3 client is ready using the imported function
verify_s3_client(s3_client, bucket_name=BUCKET, verbose=True)

# Get all TIF files using the imported function
keys = get_all_s3_keys(s3_client, BUCKET, PATH_OLD, ".tif") if s3_client else []

if keys:
    print(f"✅ Found {len(keys)} .tif files in the S3 bucket.")
else:
    print("No keys found or S3 client not initialized")
    
keys

✅ S3 client initialized successfully
   Found 68 accessible buckets
✅ S3 filesystem (fsspec) initialized
✅ S3 client ready for operations
   Bucket: nasa-disasters
   Ready to process files
✅ Found 15 .tif files in the S3 bucket.


['drcs_activations/202401_SevereWx_SoutheastUS/landsat/landsat8/LC08_colorInfrared_20240110_161317_019038.tif',
 'drcs_activations/202401_SevereWx_SoutheastUS/landsat/landsat8/LC08_colorInfrared_20240110_161341_019039.tif',
 'drcs_activations/202401_SevereWx_SoutheastUS/landsat/landsat8/LC08_colorInfrared_20240110_16145_019040.tif',
 'drcs_activations/202401_SevereWx_SoutheastUS/landsat/landsat8/LC08_naturalColor_20240110_161317_019038.tif',
 'drcs_activations/202401_SevereWx_SoutheastUS/landsat/landsat8/LC08_naturalColor_20240110_161341_019039.tif',
 'drcs_activations/202401_SevereWx_SoutheastUS/landsat/landsat8/LC08_naturalColor_20240110_16145_019040.tif',
 'drcs_activations/202401_SevereWx_SoutheastUS/landsat/landsat8/LC08_trueColor_20240110_161317_019038.tif',
 'drcs_activations/202401_SevereWx_SoutheastUS/landsat/landsat8/LC08_trueColor_20240110_161341_019039.tif',
 'drcs_activations/202401_SevereWx_SoutheastUS/landsat/landsat8/LC08_trueColor_20240110_16145_019040.tif',
 'drcs_act

# For these we can see three different types of files

We will use the same rename function and place them into the same directory


## Configure bucket and paths (no need to create session manually)

In [7]:
def return_bucket_info(config):
    """
    Extract bucket information from configuration and return as dictionary.
    
    Args:
        config: Configuration dictionary containing bucket and prefix information
    
    Returns:
        Dictionary with bucket and prefix information
    """
    # Configure bucket and paths (no need to create session manually)
    bucket_name = config["cog_data_bucket"]
    raw_data_bucket = config["raw_data_bucket"]
    raw_data_prefix = config["raw_data_prefix"]
    
    cog_data_bucket = config['cog_data_bucket']
    cog_data_prefix = config["cog_data_prefix"]
    
    print(f"Configuration loaded:")
    print(f"  Source bucket: {raw_data_bucket}")
    print(f"  Source prefix: {raw_data_prefix}")
    print(f"  Target bucket: {cog_data_bucket}")
    print(f"  Target prefix: {cog_data_prefix}")

    return {
        "bucket_name": bucket_name,
        "raw_data_bucket": raw_data_bucket,
        "raw_data_prefix": raw_data_prefix,
        "cog_data_bucket": cog_data_bucket,
        "cog_data_prefix": cog_data_prefix
    }

## Define Chunked COG Conversion Function

This function handles the conversion of files to Cloud Optimized GeoTIFFs with:
- Chunked processing to handle large files
- Memory monitoring
- Progress tracking
- Proper CRS and caching

In [8]:
# Check current cache status using the imported function
check_cache_status()

📊 Cache Status:
  - Directory: data_download/
  - Total files: 13
  - Total size: 29.40 GB

📁 Cached files (first 10):
  - drcs_activations/202401_SevereWx_SoutheastUS/landsat/landsat8/LC08_naturalColor_20240110_161317_019038.tif (167.8 MB)
  - drcs_activations/202401_SevereWx_SoutheastUS/landsat/landsat8/LC08_naturalColor_20240110_161341_019039.tif (168.2 MB)
  - drcs_activations/202401_SevereWx_SoutheastUS/landsat/landsat8/LC08_naturalColor_20240110_16145_019040.tif (168.4 MB)
  - drcs_activations/202401_SevereWx_SoutheastUS/landsat/landsat8/LC08_trueColor_20240110_161317_019038.tif (167.8 MB)
  - drcs_activations/202401_SevereWx_SoutheastUS/landsat/landsat8/LC08_trueColor_20240110_161341_019039.tif (168.2 MB)
  - drcs_activations/202401_SevereWx_SoutheastUS/landsat/landsat8/LC08_trueColor_20240110_16145_019040.tif (168.4 MB)
  - drcs_activations/202401_SevereWx_SoutheastUS/landsat/landsat9/LC09_naturalColor_20240102_161326_019038.tif (167.3 MB)
  - drcs_activations/202401_SevereWx_S

(13, 31571508092)

In [9]:
import re

def simple_process_files(keys, filter_str, rename_func, target_dir, EVENT_NAME):
    """
    Simple wrapper to process files with minimal code.
    
    Args:
        keys: List of all S3 keys
        filter_str: Can be:
            - String to filter files (e.g. 'S1_WTR')
            - Regex pattern object (e.g. re.compile(r'.*S2A.*mosaic'))
            - Callable function that returns True/False
        rename_func: Your custom rename function
        target_dir: Target directory (e.g. "Sentinel-1/opera_dswx")
        EVENT_NAME: Event name
    
    Returns:
        Processing results DataFrame
    """
    # 1. Filter files based on type of filter_str
    if callable(filter_str):
        # If it's a function
        filtered_files = [i for i in keys if filter_str(i)]
    elif hasattr(filter_str, 'search'):
        # If it's a compiled regex pattern
        filtered_files = [i for i in keys if filter_str.search(i)]
    elif isinstance(filter_str, str) and filter_str.startswith('r"') or filter_str.startswith("r'"):
        # If it's a regex string (e.g., r'pattern')
        pattern = re.compile(filter_str[2:-1])  # Remove r" or r'
        filtered_files = [i for i in keys if pattern.search(i)]
    else:
        # Default: simple string contains
        filtered_files = [i for i in keys if filter_str in i]
    
    # 2. Test renaming
    print(f"Testing filenames:")
    for f in filtered_files:
        print(f"  {rename_func(f, EVENT_NAME)}")
    
    # 3. Setup config
    config = {
        "data_acquisition_method": "s3",
        "raw_data_bucket": BUCKET,
        "raw_data_prefix": PATH_OLD,
        "cog_data_bucket": BUCKET,
        "cog_data_prefix": f'{DIR_NEW_BASE}/{target_dir}',
        "local_output_dir": f"output/{EVENT_NAME}",
        "transformation": {}
    }
    return_bucket_info(config)
    
    # 4. Process files
    print("\n" + "="*50)
    print("🌊 Processing Files (Chunked)")
    print("="*50)
    
    def chunked_converter(name, BUCKET, cog_filename, cog_data_bucket, cog_data_prefix, s3_client, local_output_dir=None):
        return convert_to_proper_CRS_and_cogify_chunked(
            name, BUCKET, cog_filename, cog_data_bucket, cog_data_prefix, s3_client, COG_PROFILE,
            local_output_dir, chunk_config=CHUNK_CONFIG
        )

    results = process_file_batch(
        file_list=filtered_files,
        s3_client=s3_client,
        config=config,
        filename_creator_func=rename_func,
        processing_func=chunked_converter,
        event_name=EVENT_NAME,
        save_metadata=True,
        save_csv=True,
        verbose=True,
        BUCKET=BUCKET
    )
    
    print_batch_summary(results)
    return results

# Process files

In [10]:
keys

['drcs_activations/202401_SevereWx_SoutheastUS/landsat/landsat8/LC08_colorInfrared_20240110_161317_019038.tif',
 'drcs_activations/202401_SevereWx_SoutheastUS/landsat/landsat8/LC08_colorInfrared_20240110_161341_019039.tif',
 'drcs_activations/202401_SevereWx_SoutheastUS/landsat/landsat8/LC08_colorInfrared_20240110_16145_019040.tif',
 'drcs_activations/202401_SevereWx_SoutheastUS/landsat/landsat8/LC08_naturalColor_20240110_161317_019038.tif',
 'drcs_activations/202401_SevereWx_SoutheastUS/landsat/landsat8/LC08_naturalColor_20240110_161341_019039.tif',
 'drcs_activations/202401_SevereWx_SoutheastUS/landsat/landsat8/LC08_naturalColor_20240110_16145_019040.tif',
 'drcs_activations/202401_SevereWx_SoutheastUS/landsat/landsat8/LC08_trueColor_20240110_161317_019038.tif',
 'drcs_activations/202401_SevereWx_SoutheastUS/landsat/landsat8/LC08_trueColor_20240110_161341_019039.tif',
 'drcs_activations/202401_SevereWx_SoutheastUS/landsat/landsat8/LC08_trueColor_20240110_16145_019040.tif',
 'drcs_act

# trueColor

In [11]:
# Define filename creator functions for different file types

def create_cog_filename(f, EVENT_NAME):
    """Extract date from filename and move to end with formatted date."""
    from pathlib import Path
    import re
    
    filename_stem = Path(f).stem
    
    # Find date pattern (8 digits starting with 20)
    date_match = re.search(r'(20\d{6})', filename_stem)
    
    if date_match:
        date_str = date_match.group(1)
        # Format date as YYYY-MM-DD
        formatted_date = f"{date_str[:4]}-{date_str[4:6]}-{date_str[6:8]}"
        
        # Remove the date from its current position
        filename_parts = filename_stem.replace(date_str + '_', '')
        
        # Create new filename with EVENT_NAME + parts + formatted date + day
        cog_filename = f'{EVENT_NAME}_{filename_parts}_{formatted_date}_day.tif'
    else:
        # No date found, just add event name
        cog_filename = f'{EVENT_NAME}_{filename_stem}_day.tif'
    
    return cog_filename


filter_str = "trueColor"

# Test functions
print("Testing WM filename:")
filter_ =  [f for f in keys if filter_str in f]

for idx,i in enumerate(filter_):
    test_wm = create_cog_filename(filter_[idx], EVENT_NAME)
    print(f"  {test_wm}")




Testing WM filename:
  202401_SevereWx_SoutheastUS_LC08_trueColor_161317_019038_2024-01-10_day.tif
  202401_SevereWx_SoutheastUS_LC08_trueColor_161341_019039_2024-01-10_day.tif
  202401_SevereWx_SoutheastUS_LC08_trueColor_16145_019040_2024-01-10_day.tif
  202401_SevereWx_SoutheastUS_LC09_trueColor_161326_019038_2024-01-02_day.tif
  202401_SevereWx_SoutheastUS_LC09_trueColor_161350_019039_2024-01-02_day.tif


In [12]:
# Process S1 WTR files
results1 = simple_process_files(keys=keys, 
                                filter_str = filter_str, 
                                rename_func = create_cog_filename, 
                                target_dir = "Landsat/trueColor", 
                                EVENT_NAME = EVENT_NAME)


Testing filenames:
  202401_SevereWx_SoutheastUS_LC08_trueColor_161317_019038_2024-01-10_day.tif
  202401_SevereWx_SoutheastUS_LC08_trueColor_161341_019039_2024-01-10_day.tif
  202401_SevereWx_SoutheastUS_LC08_trueColor_16145_019040_2024-01-10_day.tif
  202401_SevereWx_SoutheastUS_LC09_trueColor_161326_019038_2024-01-02_day.tif
  202401_SevereWx_SoutheastUS_LC09_trueColor_161350_019039_2024-01-02_day.tif
Configuration loaded:
  Source bucket: nasa-disasters
  Source prefix: drcs_activations/202401_SevereWx_SoutheastUS/landsat
  Target bucket: nasa-disasters
  Target prefix: drcs_activations_new/Landsat/trueColor

🌊 Processing Files (Chunked)
✅ Local output directory ready: output/202401_SevereWx_SoutheastUS

[1/5] Processing: drcs_activations/202401_SevereWx_SoutheastUS/landsat/landsat8/LC08_trueColor_20240110_161317_019038.tif
   Output filename: 202401_SevereWx_SoutheastUS_LC08_trueColor_161317_019038_2024-01-10_day.tif
   [MEMORY] Initial: 288.4 MB
   [CACHE HIT] Using cached file: 

   [BAND 2/3] Processing...


Band 2:  65%|██████▌   | 47/72 [00:01<00:00, 25.37chunks/s]


   [MEMORY] High usage: 584.0 MB, forcing cleanup...


Band 2:  82%|████████▏ | 59/72 [00:02<00:00, 25.77chunks/s]

...

   [VERIFY] Checking reprojected data...
   [VERIFY] RGB file without nodata - all pixel values are valid
   [VERIFY] Band 1: min=28, max=255, center sample non-zero=1000000/1000000
            Estimated data coverage: 100.0% (from distributed samples)
   [VERIFY] Band 2: min=48, max=255, center sample non-zero=1000000/1000000
            Estimated data coverage: 100.0% (from distributed samples)
   [VERIFY] Band 3: min=88, max=255, center sample non-zero=1000000/1000000
            Estimated data coverage: 100.0% (from distributed samples)
   [COGIFY] Creating COG from reprojected file...
   [NODATA] Data type: uint8
   [NODATA] Using nodata value 0 for uint8 data
   [PREDICTOR] Data type: uint8, using PREDICTOR=2
   [WRITE] Writing temporary GeoTIFF with chunked processing...


Reading input: /tmp/tmpzmd2n3fv_temp.tif



   [CONVERT] Converting GeoTIFF to COG...
   [CONVERT] Using rio-cogeo for conversion...


Adding overviews...
Updating dataset tags...
Writing output to: /tmp/tmp6x_fg6rj.tif


   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - Compression 'zstd' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/Landsat/trueColor/202401_SevereWx_SoutheastUS_LC09_trueColor_161350_019039_2024-01-02_day.tif
   [MEMORY] Final: 949.9 MB (Change: +7.6 MB)
✅ Chunked COG conversion function defined with memory-efficient processing
   ✅ Generated and saved COG: 202401_SevereWx_SoutheastUS_LC09_trueColor_161350_019039_2024-01-02_day.tif

✅ Batch processing complete: 5 files processed
📊 Uploaded metadata to s3://nasa-disasters/drcs_activations_new/Landsat/trueColor/metadata.json
📝 Saved processing log to s3://nasa-disasters/drcs_activations_new/Landsat/trueColor/files_converted.csv
📁 COGs saved locally to: output/202401_SevereWx_SoutheastUS

📊 BATCH PROCESSING SUMMARY
Total files processed: 5
Successful: 5
Failed: 0
Success rate: 100.0%
Timestamp: 2025-09-09T15:52:46.253970


# naturalColor

In [13]:
# Define filename creator functions for different file types
filter_str = 'naturalColor'

# Test functions
print("Testing WM filename:")
filter_ =  [f for f in keys if filter_str in f]

for idx,i in enumerate(filter_):
    test_wm = create_cog_filename(filter_[idx], EVENT_NAME)
    print(f"  {test_wm}")




Testing WM filename:
  202401_SevereWx_SoutheastUS_LC08_naturalColor_161317_019038_2024-01-10_day.tif
  202401_SevereWx_SoutheastUS_LC08_naturalColor_161341_019039_2024-01-10_day.tif
  202401_SevereWx_SoutheastUS_LC08_naturalColor_16145_019040_2024-01-10_day.tif
  202401_SevereWx_SoutheastUS_LC09_naturalColor_161326_019038_2024-01-02_day.tif
  202401_SevereWx_SoutheastUS_LC09_naturalColor_161350_019039_2024-01-02_day.tif


In [14]:
# Process S1 WTR files
results1 = simple_process_files(keys=keys, 
                                filter_str = filter_str, 
                                rename_func = create_cog_filename, 
                                target_dir = "Landsat/naturalColor", 
                                EVENT_NAME = EVENT_NAME)


Testing filenames:
  202401_SevereWx_SoutheastUS_LC08_naturalColor_161317_019038_2024-01-10_day.tif
  202401_SevereWx_SoutheastUS_LC08_naturalColor_161341_019039_2024-01-10_day.tif
  202401_SevereWx_SoutheastUS_LC08_naturalColor_16145_019040_2024-01-10_day.tif
  202401_SevereWx_SoutheastUS_LC09_naturalColor_161326_019038_2024-01-02_day.tif
  202401_SevereWx_SoutheastUS_LC09_naturalColor_161350_019039_2024-01-02_day.tif
Configuration loaded:
  Source bucket: nasa-disasters
  Source prefix: drcs_activations/202401_SevereWx_SoutheastUS/landsat
  Target bucket: nasa-disasters
  Target prefix: drcs_activations_new/Landsat/naturalColor

🌊 Processing Files (Chunked)
✅ Local output directory ready: output/202401_SevereWx_SoutheastUS

[1/5] Processing: drcs_activations/202401_SevereWx_SoutheastUS/landsat/landsat8/LC08_naturalColor_20240110_161317_019038.tif
   Output filename: 202401_SevereWx_SoutheastUS_LC08_naturalColor_161317_019038_2024-01-10_day.tif
   [MEMORY] Initial: 951.1 MB
   [CACHE 

   [BAND 2/3] Processing...


   [BAND 3/3] Processing...


   [VERIFY] Checking reprojected data...
   [VERIFY] RGB file without nodata - all pixel values are valid
   [VERIFY] Band 1: min=1, max=255, center sample non-zero=1000000/1000000
            Estimated data coverage: 100.0% (from distributed samples)
   [VERIFY] Band 2: min=1, max=255, center sample non-zero=1000000/1000000
            Estimated data coverage: 100.0% (from distributed samples)
   [VERIFY] Band 3: min=26, max=255, center sample non-zero=1000000/1000000
            Estimated data coverage: 100.0% (from distributed samples)
   [COGIFY] Creating COG from reprojected file...
   [NODATA] Data type: uint8
   [NODATA] Using nodata value 0 for uint8 data
   [PREDICTOR] Data type: uint8, using PREDICTOR=2
   [WRITE] Writing temporary GeoTIFF with chunked processing...


Reading input: /tmp/tmpfi54ifo6_temp.tif



...

   [VERIFY] Checking reprojected data...
   [VERIFY] RGB file without nodata - all pixel values are valid
   [VERIFY] Band 1: min=1, max=255, center sample non-zero=1000000/1000000
            Estimated data coverage: 100.0% (from distributed samples)
   [VERIFY] Band 2: min=1, max=255, center sample non-zero=1000000/1000000
            Estimated data coverage: 100.0% (from distributed samples)
   [VERIFY] Band 3: min=32, max=255, center sample non-zero=1000000/1000000
            Estimated data coverage: 100.0% (from distributed samples)
   [COGIFY] Creating COG from reprojected file...
   [NODATA] Data type: uint8
   [NODATA] Using nodata value 0 for uint8 data
   [PREDICTOR] Data type: uint8, using PREDICTOR=2
   [WRITE] Writing temporary GeoTIFF with chunked processing...


Reading input: /tmp/tmphhm2tzco_temp.tif



   [CONVERT] Converting GeoTIFF to COG...
   [CONVERT] Using rio-cogeo for conversion...


Adding overviews...
Updating dataset tags...
Writing output to: /tmp/tmpcz4c__ik.tif


   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - Compression 'zstd' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/Landsat/naturalColor/202401_SevereWx_SoutheastUS_LC09_naturalColor_161350_019039_2024-01-02_day.tif
   [MEMORY] Final: 1013.2 MB (Change: -25.0 MB)
✅ Chunked COG conversion function defined with memory-efficient processing
   ✅ Generated and saved COG: 202401_SevereWx_SoutheastUS_LC09_naturalColor_161350_019039_2024-01-02_day.tif

✅ Batch processing complete: 5 files processed
📊 Uploaded metadata to s3://nasa-disasters/drcs_activations_new/Landsat/naturalColor/metadata.json
📝 Saved processing log to s3://nasa-disasters/drcs_activations_new/Landsat/naturalColor/files_converted.csv
📁 COGs saved locally to: output/202401_SevereWx_SoutheastUS

📊 BATCH PROCESSING SUMMARY
Total files processed: 5
Successful: 5
Failed: 0
Success rate: 100.0%
Timestamp: 2025-09-09T

In [15]:
keys

['drcs_activations/202401_SevereWx_SoutheastUS/landsat/landsat8/LC08_colorInfrared_20240110_161317_019038.tif',
 'drcs_activations/202401_SevereWx_SoutheastUS/landsat/landsat8/LC08_colorInfrared_20240110_161341_019039.tif',
 'drcs_activations/202401_SevereWx_SoutheastUS/landsat/landsat8/LC08_colorInfrared_20240110_16145_019040.tif',
 'drcs_activations/202401_SevereWx_SoutheastUS/landsat/landsat8/LC08_naturalColor_20240110_161317_019038.tif',
 'drcs_activations/202401_SevereWx_SoutheastUS/landsat/landsat8/LC08_naturalColor_20240110_161341_019039.tif',
 'drcs_activations/202401_SevereWx_SoutheastUS/landsat/landsat8/LC08_naturalColor_20240110_16145_019040.tif',
 'drcs_activations/202401_SevereWx_SoutheastUS/landsat/landsat8/LC08_trueColor_20240110_161317_019038.tif',
 'drcs_activations/202401_SevereWx_SoutheastUS/landsat/landsat8/LC08_trueColor_20240110_161341_019039.tif',
 'drcs_activations/202401_SevereWx_SoutheastUS/landsat/landsat8/LC08_trueColor_20240110_16145_019040.tif',
 'drcs_act

# colorInfrared

In [16]:
# Define filename creator functions for different file types

filter_str = 'colorInfrared'

# Test functions
print("Testing WM filename:")
filter_ =  [f for f in keys if filter_str in f]

for idx,i in enumerate(filter_):
    test_wm = create_cog_filename(filter_[idx], EVENT_NAME)
    print(f"  {test_wm}")

Testing WM filename:
  202401_SevereWx_SoutheastUS_LC08_colorInfrared_161317_019038_2024-01-10_day.tif
  202401_SevereWx_SoutheastUS_LC08_colorInfrared_161341_019039_2024-01-10_day.tif
  202401_SevereWx_SoutheastUS_LC08_colorInfrared_16145_019040_2024-01-10_day.tif
  202401_SevereWx_SoutheastUS_LC09_colorInfrared_161326_019038_2024-01-02_day.tif
  202401_SevereWx_SoutheastUS_LC09_colorInfrared_161350_019039_2024-01-02_day.tif


In [17]:
# Process S1 WTR files

results1 = simple_process_files(keys=keys, 
                                filter_str = filter_str, 
                                rename_func = create_cog_filename, 
                                target_dir = "Landsat/colorInfrared", 
                                EVENT_NAME = EVENT_NAME)


Testing filenames:
  202401_SevereWx_SoutheastUS_LC08_colorInfrared_161317_019038_2024-01-10_day.tif
  202401_SevereWx_SoutheastUS_LC08_colorInfrared_161341_019039_2024-01-10_day.tif
  202401_SevereWx_SoutheastUS_LC08_colorInfrared_16145_019040_2024-01-10_day.tif
  202401_SevereWx_SoutheastUS_LC09_colorInfrared_161326_019038_2024-01-02_day.tif
  202401_SevereWx_SoutheastUS_LC09_colorInfrared_161350_019039_2024-01-02_day.tif
Configuration loaded:
  Source bucket: nasa-disasters
  Source prefix: drcs_activations/202401_SevereWx_SoutheastUS/landsat
  Target bucket: nasa-disasters
  Target prefix: drcs_activations_new/Landsat/colorInfrared

🌊 Processing Files (Chunked)
✅ Local output directory ready: output/202401_SevereWx_SoutheastUS

[1/5] Processing: drcs_activations/202401_SevereWx_SoutheastUS/landsat/landsat8/LC08_colorInfrared_20240110_161317_019038.tif
   Output filename: 202401_SevereWx_SoutheastUS_LC08_colorInfrared_161317_019038_2024-01-10_day.tif
   [MEMORY] Initial: 974.2 MB
  

   [BAND 2/3] Processing...


   [BAND 3/3] Processing...


   [VERIFY] Checking reprojected data...
   [VERIFY] RGB file without nodata - all pixel values are valid
   [VERIFY] Band 1: min=1, max=255, center sample non-zero=1000000/1000000
            Estimated data coverage: 100.0% (from distributed samples)
   [VERIFY] Band 2: min=34, max=255, center sample non-zero=1000000/1000000
            Estimated data coverage: 100.0% (from distributed samples)
   [VERIFY] Band 3: min=53, max=255, center sample non-zero=1000000/1000000
            Estimated data coverage: 100.0% (from distributed samples)
   [COGIFY] Creating COG from reprojected file...
   [NODATA] Data type: uint8
   [NODATA] Using nodata value 0 for uint8 data
   [PREDICTOR] Data type: uint8, using PREDICTOR=2
   [WRITE] Writing temporary GeoTIFF with chunked processing...


Reading input: /tmp/tmph72y__93_temp.tif



...

   [VERIFY] Checking reprojected data...
   [VERIFY] RGB file without nodata - all pixel values are valid
   [VERIFY] Band 1: min=1, max=255, center sample non-zero=1000000/1000000
            Estimated data coverage: 100.0% (from distributed samples)
   [VERIFY] Band 2: min=36, max=255, center sample non-zero=1000000/1000000
            Estimated data coverage: 100.0% (from distributed samples)
   [VERIFY] Band 3: min=57, max=255, center sample non-zero=1000000/1000000
            Estimated data coverage: 100.0% (from distributed samples)
   [COGIFY] Creating COG from reprojected file...
   [NODATA] Data type: uint8
   [NODATA] Using nodata value 0 for uint8 data
   [PREDICTOR] Data type: uint8, using PREDICTOR=2
   [WRITE] Writing temporary GeoTIFF with chunked processing...


Reading input: /tmp/tmpwm6g28kv_temp.tif



   [CONVERT] Converting GeoTIFF to COG...
   [CONVERT] Using rio-cogeo for conversion...


Adding overviews...
Updating dataset tags...
Writing output to: /tmp/tmpoxmt60eq.tif


   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - Compression 'zstd' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/Landsat/colorInfrared/202401_SevereWx_SoutheastUS_LC09_colorInfrared_161350_019039_2024-01-02_day.tif
   [MEMORY] Final: 982.3 MB (Change: -10.6 MB)
✅ Chunked COG conversion function defined with memory-efficient processing
   ✅ Generated and saved COG: 202401_SevereWx_SoutheastUS_LC09_colorInfrared_161350_019039_2024-01-02_day.tif

✅ Batch processing complete: 5 files processed
📊 Uploaded metadata to s3://nasa-disasters/drcs_activations_new/Landsat/colorInfrared/metadata.json
📝 Saved processing log to s3://nasa-disasters/drcs_activations_new/Landsat/colorInfrared/files_converted.csv
📁 COGs saved locally to: output/202401_SevereWx_SoutheastUS

📊 BATCH PROCESSING SUMMARY
Total files processed: 5
Successful: 5
Failed: 0
Success rate: 100.0%
Timestamp: 2025-09

## Check STATUS of file conversion and upload

<a href="https://data.disasters.openveda.cloud/browseui/browseui/#drcs_activations_new/" target="_blank" rel="noopener noreferrer" style="color: blue; font-size: 20px;">Disasters Bucket</a> -- You can view that the files actually made it to their correct destination.

## Memory Usage Summary

You can check the final memory usage and cleanup

In [18]:
# Final memory cleanup and report
gc.collect()
final_memory = get_memory_usage()
print(f"\n📊 Memory Usage Summary:")
print(f"  Current memory usage: {final_memory:.1f} MB")
print(f"  Available memory: {psutil.virtual_memory().available / 1024 / 1024:.1f} MB")
print(f"  Memory percent used: {psutil.virtual_memory().percent:.1f}%")


📊 Memory Usage Summary:
  Current memory usage: 982.3 MB
  Available memory: 27848.8 MB
  Memory percent used: 11.9%
